<a href="https://colab.research.google.com/github/fareed-asif/Student_Performance_Analysis/blob/main/SP24_BAI_031(PFAI_LAB_ASS_3_TASK3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/StudentsPerformance.csv")
score_cols = ['math score', 'reading score', 'writing score']
print("Original data loaded. Shape:", df.shape)

Original data loaded. Shape: (1000, 8)


In [2]:
messy = df.copy()
np.random.seed(42)

# Add 50 missing math scores
idx_missing = np.random.choice(messy.index, 50, replace=False)
messy.loc[idx_missing, 'math score'] = np.nan

# Add 30 missing gender values
idx_missing2 = np.random.choice(messy.index, 30, replace=False)
messy.loc[idx_missing2, 'gender'] = np.nan

# Add 20 duplicate rows
messy = pd.concat([messy, messy.sample(20)], ignore_index=True)

# Add impossible/extreme outlier scores
messy.loc[5, 'math score'] = 150    # max possible is 100!
messy.loc[10, 'reading score'] = -10  # can't be negative!
messy.loc[15, 'writing score'] = 999  # way too high!

# Make gender text UPPERCASE (inconsistent)
messy['gender'] = messy['gender'].str.upper()

print("Messy dataset created!")
print("Shape:", messy.shape)

Messy dataset created!
Shape: (1020, 8)


In [3]:
print("=== MISSING VALUES ===")
print(messy.isnull().sum())
print(f"\nTotal missing cells: {messy.isnull().sum().sum()}")
print(f"Total rows: {len(messy)}")

=== MISSING VALUES ===
gender                         31
race/ethnicity                  0
parental level of education     0
lunch                           0
test preparation course         0
math score                     51
reading score                   0
writing score                   0
dtype: int64

Total missing cells: 82
Total rows: 1020


In [4]:
print("=== DUPLICATES ===")
print(f"Number of duplicate rows: {messy.duplicated().sum()}")
print("\nExample duplicate rows:")
print(messy[messy.duplicated()].head(3))

=== DUPLICATES ===
Number of duplicate rows: 20

Example duplicate rows:
      gender race/ethnicity parental level of education         lunch  \
1000    MALE        group C           bachelor's degree      standard   
1001  FEMALE        group A                 high school  free/reduced   
1002  FEMALE        group B           bachelor's degree      standard   

     test preparation course  math score  reading score  writing score  
1000               completed        94.0             90             91  
1001               completed        77.0             88             85  
1002                    none        61.0             72             70  


In [5]:
print("=== OUTLIERS (IQR Method) ===")
for col in score_cols:
    Q1 = messy[col].quantile(0.25)
    Q3 = messy[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = messy[(messy[col] < lower) | (messy[col] > upper)]
    print(f"{col}: {len(outliers)} outliers | Valid range: {lower:.1f} to {upper:.1f}")

=== OUTLIERS (IQR Method) ===
math score: 8 outliers | Valid range: 27.0 to 107.0
reading score: 6 outliers | Valid range: 27.5 to 111.5
writing score: 6 outliers | Valid range: 25.9 to 110.9


In [6]:
print("=== GENDER VALUES ===")
print(messy['gender'].value_counts(dropna=False))
print("\nUnique values:", messy['gender'].unique())

=== GENDER VALUES ===
gender
FEMALE    512
MALE      477
NaN        31
Name: count, dtype: int64

Unique values: ['FEMALE' 'MALE' nan]


In [7]:
cleaned = messy.copy()

# Convert gender to lowercase
cleaned['gender'] = cleaned['gender'].str.lower()

print("Gender values BEFORE:", messy['gender'].unique())
print("Gender values AFTER: ", cleaned['gender'].unique())

Gender values BEFORE: ['FEMALE' 'MALE' nan]
Gender values AFTER:  ['female' 'male' nan]


In [8]:
# Fill missing numbers with the median
for col in score_cols:
    median_val = cleaned[col].median()
    missing_count = cleaned[col].isnull().sum()
    cleaned[col].fillna(median_val, inplace=True)
    print(f"{col}: filled {missing_count} missing values with median={median_val:.1f}")

# Fill missing gender with mode (most common)
mode_gender = cleaned['gender'].mode()[0]
missing_gender = cleaned['gender'].isnull().sum()
cleaned['gender'].fillna(mode_gender, inplace=True)
print(f"\ngender: filled {missing_gender} missing values with mode='{mode_gender}'")

print("\nMissing values remaining:", cleaned.isnull().sum().sum())

math score: filled 51 missing values with median=66.0
reading score: filled 0 missing values with median=70.0
writing score: filled 0 missing values with median=69.0

gender: filled 31 missing values with mode='female'

Missing values remaining: 0


/tmp/ipykernel_2604/2768905018.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cleaned[col].fillna(median_val, inplace=True)
/tmp/ipykernel_2604/2768905018.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

In [9]:
print(f"Rows BEFORE removing duplicates: {len(cleaned)}")
cleaned.drop_duplicates(inplace=True)
cleaned.reset_index(drop=True, inplace=True)
print(f"Rows AFTER removing duplicates:  {len(cleaned)}")
print(f"Duplicates removed: {1020 - len(cleaned)}")

Rows BEFORE removing duplicates: 1020
Rows AFTER removing duplicates:  1000
Duplicates removed: 20


In [10]:
print("Fixing outliers using IQR clipping...")
for col in score_cols:
    Q1 = cleaned[col].quantile(0.25)
    Q3 = cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before_count = ((cleaned[col] < lower) | (cleaned[col] > upper)).sum()
    cleaned[col] = cleaned[col].clip(lower, upper)
    print(f"{col}: clipped to [{lower:.1f}, {upper:.1f}] — fixed {before_count} values")

print("\nOutlier check after fixing:")
for col in score_cols:
    print(f"{col}: min={cleaned[col].min():.1f}, max={cleaned[col].max():.1f}")

Fixing outliers using IQR clipping...
math score: clipped to [31.0, 103.0] — fixed 14 values
reading score: clipped to [29.0, 109.0] — fixed 7 values
writing score: clipped to [25.9, 110.9] — fixed 6 values

Outlier check after fixing:
math score: min=31.0, max=103.0
reading score: min=29.0, max=100.0
writing score: min=25.9, max=110.9


In [11]:
cleaned.to_csv("/content/cleaned_students.csv", index=False)
print("Cleaned file saved!")

Cleaned file saved!


In [12]:
print("====== BEFORE vs AFTER SUMMARY ======")
print(f"Shape:    {messy.shape}  →  {cleaned.shape}")
print(f"Missing:  {messy.isnull().sum().sum()}  →  {cleaned.isnull().sum().sum()}")
print(f"Duplicates: {messy.duplicated().sum()}  →  {cleaned.duplicated().sum()}")
print("\nScore ranges BEFORE:")
print(messy[score_cols].agg(['min','max']))
print("\nScore ranges AFTER:")
print(cleaned[score_cols].agg(['min','max']))

====== BEFORE vs AFTER SUMMARY ======
Shape:    (1020, 8)  →  (1000, 8)
Missing:  82  →  0
Duplicates: 20  →  0

Score ranges BEFORE:
     math score  reading score  writing score
min         8.0            -10             10
max       150.0            100            999

Score ranges AFTER:
     math score  reading score  writing score
min        31.0             29         25.875
max       103.0            100        110.875
